# Cross-study gene expression comparison

Compare expression of a target gene (and optional housekeepers) across all datasets.
Uses the `/gene_stats` endpoint to fetch per-gene statistics in a single lightweight call per dataset — no need to download all cells.

In [ ]:
try:
    import micropip
    await micropip.install(['antclient', 'pandas', 'matplotlib'])
except:
    pass  # already installed in standard Python
print('ready')

In [ ]:
from antclient import AnthiveClient
import pandas as pd
import matplotlib.pyplot as plt

try:
    import js
    api_url = f"{js.location.origin}/api/"
except:
    api_url = None

client = AnthiveClient(api_url)
print(f'Connected to {client.base_url}')

## Configuration

`target_gene` is the gene you're interested in. `housekeepers` are reference genes used to normalise or provide context (set to `[]` to skip). Datasets without a `default_layer` are skipped automatically.

In [ ]:
target_gene  = 'Apoe'
housekeepers = ['Actb', 'Gapdh']  # set [] to skip

## Collect statistics across studies

For each dataset we make one API call returning mean, std, pct_nonzero etc. — no large data transfer.

In [ ]:
databases = client.get_databases()
print(f'Querying {len(databases)} databases...')

genes_to_query = [target_gene] + housekeepers
rows = []
for db in databases:
    db_id = db['id']
    layer = db.get('default_layer')
    if layer is None:
        print(f'  skipped {db_id}: no default_layer')
        continue
    try:
        stats = client.get_gene_stats(db_id, genes_to_query, layer=layer)
        for s in stats:
            s['db_id']      = db_id
            s['title']      = db.get('title', db_id)
            s['n_cells_db'] = db.get('n_cells', 0)
            rows.append(s)
    except Exception as e:
        print(f'  skipped {db_id}: {e}')

df = pd.DataFrame(rows)
print(f'\n{len(df)} rows ({df["db_id"].nunique()} databases)')
df.head()

## Plot: mean expression and detection rate

Left panel: mean expression ± std across all cells (zeros included).
Right panel: % cells expressing the gene.

In [ ]:
# Filter to target gene only; skip databases where gene was not found
tgt = df[(df['gene'] == target_gene) & (~df.get('error', pd.Series(dtype=str)).notna() if 'error' in df.columns else df['n_nonzero'] >= 0)].copy()
if 'error' in tgt.columns:
    tgt = tgt[tgt['error'].isna()]
tgt = tgt.sort_values('mean', ascending=True)  # ascending so top = highest

fig, axes = plt.subplots(1, 2, figsize=(13, max(3, len(tgt) * 0.38)))

# Left: mean ± std
ax = axes[0]
ax.barh(tgt['title'], tgt['mean'], xerr=tgt['std'],
        color='steelblue', alpha=0.75, capsize=3, ecolor='gray')
ax.set_xlabel('Mean expression (all cells)')
ax.set_title(f'{target_gene} mean expression')

# Right: detection rate
ax = axes[1]
ax.barh(tgt['title'], tgt['pct_nonzero'], color='tomato', alpha=0.75)
ax.set_xlabel('% expressing cells')
ax.set_title(f'{target_gene} detection rate')

plt.tight_layout()
plt.show()

## Housekeepers: normalised expression

Express the target gene relative to each housekeeper's mean (ratio). This helps correct for differences in sequencing depth or normalisation across studies.

In [ ]:
if not housekeepers:
    print('No housekeepers configured — skip this cell')
else:
    # Pivot to wide format: one row per database
    wide = df.pivot_table(index='db_id', columns='gene', values='mean', aggfunc='first').reset_index()
    if target_gene not in wide.columns:
        print(f'{target_gene} not found in any database')
    else:
        fig, axes = plt.subplots(1, len(housekeepers), figsize=(5 * len(housekeepers), 4), squeeze=False)
        for i, hk in enumerate(housekeepers):
            ax = axes[0][i]
            if hk not in wide.columns:
                ax.set_title(f'{hk} — not found')
                continue
            ratio = wide[target_gene] / wide[hk].replace(0, float('nan'))
            order = ratio.sort_values(ascending=True)
            ax.barh(wide.loc[order.index, 'db_id'], order.values, color='mediumpurple', alpha=0.75)
            ax.set_xlabel(f'{target_gene} / {hk}')
            ax.set_title(f'Relative to {hk}')
        plt.tight_layout()
        plt.show()